# Household Power Dataset EDA aligned with `dataset.py`

Independent public time-series benchmark added for reviewer **R2.6**
(*UCI Individual Household Electric Power Consumption*, dataset id 235,
DOI 10.24432/C58K54, CC BY 4.0).

This notebook walks through **exactly** the pipeline in
`data/household_power/dataset.py`, so the household benchmark is the *only*
changed variable versus network monitoring: two continuous signals, hourly
resampling, `[0,1]` MinMax normalisation, length-24 windows, one-step-ahead
targets, and the same GRU forecaster.

**Where to get the data.** Download the zip from
<https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption>,
unzip `household_power_consumption.txt` into `data/household_power/`. Or set
`USE_UCIMLREPO = True` below after `pip install ucimlrepo`.

## 1. Imports and configuration

In [1]:
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# --- constants kept identical to network_monitoring / dataset.py ---
CFG_DATA_SPLIT   = 0.8
CFG_SEQUENCE_LEN = 24            # 24 hourly steps = one day  (== network SEQUENCE_LEN)
CFG_SAMPLING_RATE = 1
CFG_STRIDE       = 1
CFG_BATCH_SIZE   = 1
CFG_RESAMPLE     = "60min"       # "60min" avoids the pandas 'H'/'h' deprecation

FEATURES = ["Global_active_power", "Global_reactive_power"]  # two related kW signals

# point this at the unzipped file, or flip USE_UCIMLREPO
DATA_DIR   = Path("data/household_power")
FILE_PATH  = DATA_DIR.joinpath("household_power_consumption.txt")
USE_UCIMLREPO = True

pd.set_option("display.max_columns", 20)
print("sequence length:", CFG_SEQUENCE_LEN, "| features:", FEATURES, "| resample:", CFG_RESAMPLE)

sequence length: 24 | features: ['Global_active_power', 'Global_reactive_power'] | resample: 60min


## 2. Load raw data and inspect global structure

The raw file is semicolon-separated, minute-level, ~2.075M rows over 47 months.
Missing measurements appear as `?` (or empty fields) and are read as `NaN`.

In [3]:
if not FILE_PATH.exists():
    if USE_UCIMLREPO:
        from ucimlrepo import fetch_ucirepo

        print(f"Dataset not found at {FILE_PATH}. Downloading...")
        raw_df = fetch_ucirepo(id=235).data.features.copy()

        # Ensure parent directory exists
        FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

        # Save for future use
        raw_df.to_csv(FILE_PATH, sep=";", index=False)
        print(f"Dataset saved to {FILE_PATH}")
    else:
        raise FileNotFoundError(
            f"Dataset not found at {FILE_PATH}. "
            "Either set USE_UCIMLREPO=True or place the dataset there."
        )

# Always load from the local file
raw_df = pd.read_csv(FILE_PATH, sep=";", na_values=["?"], low_memory=False)

print("raw shape:", raw_df.shape)
print("columns:", list(raw_df.columns))
raw_df.head()

Dataset not found at data\household_power\household_power_consumption.txt. Downloading...


D:\Study\PhD\1Year\Publications\ScientificReports\26_27\2025-Kokin-Privacy-Preserving-in-Federated-Learning\venv\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Dataset saved to data\household_power\household_power_consumption.txt
raw shape: (2075259, 9)
columns: ['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 3. The nine source columns; we select two power features

`Date, Time, Global_active_power, Global_reactive_power, Voltage, Global_intensity,
Sub_metering_1, Sub_metering_2, Sub_metering_3`. We keep the two global power
signals (both in kW) — the analogue of the two protocol-count signals in network
monitoring.

In [ ]:
print("all columns and dtypes:")
print(raw_df.dtypes)
print("\nselected features:", FEATURES)
# quick numeric coercion just for the summary here
_summary = raw_df[FEATURES].apply(pd.to_numeric, errors="coerce")
_summary.describe()

## 4. Reproduce `dataset.py`: build the datetime index and select two features

`Date` (dd/mm/yyyy) + `Time` (hh:mm:ss) are combined into a single datetime index;
the two features are coerced to float (turning any `?`/blank into `NaN`).

In [ ]:
dt = pd.to_datetime(
    raw_df["Date"].astype(str) + " " + raw_df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S", errors="coerce",
)
data_df = raw_df.set_index(dt)
data_df = data_df[FEATURES].apply(pd.to_numeric, errors="coerce")
data_df = data_df[~data_df.index.isna()].sort_index()

print("index span:", data_df.index.min(), "->", data_df.index.max())
print("minute-level rows:", len(data_df))
data_df.head()

## 5. Timestamp continuity (native one-minute sampling)

In [ ]:
intervals = data_df.index.to_series().diff().dropna()
print("most common interval:", intervals.mode().iloc[0])
print("interval value counts (top 5):")
print(intervals.value_counts().head(5))

## 6. Missing-value handling

Around 1.25% of rows are missing. We quantify it before resampling; the hourly
mean plus time interpolation (next section) absorbs the gaps.

In [ ]:
missing = data_df.isna().sum().to_frame("missing_minutes")
missing["pct"] = (missing["missing_minutes"] / len(data_df) * 100).round(3)
missing

## 7. Resample to hourly and interpolate; visualise raw vs. hourly

Minute-level -> hourly mean gives ~34k points and makes period 24 = one day,
matching `SEQUENCE_LEN`. Small residual gaps are filled by time interpolation.

In [ ]:
hourly = data_df.resample(CFG_RESAMPLE).mean()
hourly = hourly.interpolate(method="time").dropna()
print("hourly rows:", len(hourly), "| span:", hourly.index.min(), "->", hourly.index.max())

# one week: raw minute series vs hourly mean, for the active-power feature
wk = slice("2007-01-01", "2007-01-08")
fig, ax = plt.subplots(figsize=(11, 3.2))
data_df.loc[wk, FEATURES[0]].plot(ax=ax, lw=0.4, alpha=0.5, label="minute")
hourly.loc[wk, FEATURES[0]].plot(ax=ax, lw=1.6, label="hourly mean")
ax.set_title(f"{FEATURES[0]} — one week (minute vs hourly)"); ax.set_ylabel("kW"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Train/test split (80/20, chronological — no shuffling of time)

In [ ]:
values = hourly.values.astype("float32")
rows = len(values)
train_size = int(rows * CFG_DATA_SPLIT)
raw_train, raw_test = values[:train_size], values[train_size:]
print(f"train rows: {len(raw_train)}  ({hourly.index[0]} -> {hourly.index[train_size-1]})")
print(f"test  rows: {len(raw_test)}  ({hourly.index[train_size]} -> {hourly.index[-1]})")

## 9. Transform and normalise (MinMax to [0,1], scaler fit on train only)

The GRU output is sigmoid, so targets must live in `[0,1]`. The same positivity
guard as network monitoring maps exact zeros to a tiny positive value.

In [ ]:
def _normalize(data, scaler=None):
    if data.ndim == 1:
        data = data.reshape(-1, 1)
    if scaler is None:
        scaler = MinMaxScaler(feature_range=(0, 1))
        norm = scaler.fit_transform(data)
        norm = np.where(norm <= 0, 1e-3, norm)
    else:
        norm = scaler.transform(data)
    return norm, scaler

train_n, scaler = _normalize(raw_train)
test_n, _ = _normalize(raw_test, scaler)
print("train range:", train_n.min(axis=0), "->", train_n.max(axis=0))
print("test  range:", test_n.min(axis=0), "->", test_n.max(axis=0), "(test may slightly exceed [0,1])")

## 10. STL decomposition — exploratory only (not applied in the pipeline)

Network monitoring applies STL to *denoise* one noisy protocol count (dropping
the residual). The hourly power series is already smooth, so `dataset.py` does
**not** drop any component — the GRU is trained on the full signal. We show the
decomposition here purely to characterise the strong daily seasonality; nothing
below feeds back into the pipeline. (If strict pipeline identity with network is
preferred, an STL step retaining trend+seasonal+resid is a lossless drop-in.)

In [ ]:
from statsmodels.tsa.seasonal import STL
res = STL(hourly[FEATURES[0]], period=CFG_SEQUENCE_LEN).fit()
fig = res.plot(); fig.set_size_inches(11, 6.5)
plt.tight_layout(); plt.show()
print("residual std / signal std:", round(res.resid.std() / hourly[FEATURES[0]].std(), 3))

## 11. Horizontal FL representation

Household power is **horizontal-only**: the two power features are modelled
jointly and clients hold disjoint time spans (IID K-Fold split, done in
`config.py`). Unlike network monitoring, there is no per-feature vertical split —
`dataset_loader.py` therefore returns `is_vertical = False` for this dataset.

In [ ]:
print("features modelled jointly:", FEATURES)
print("vertical partitioning: NOT used for household_power (horizontal only)")
print("client partitioning (IID K-Fold) is applied later in config.prepare_partitions")

## 12. Time-series window generation (length 24, one-step-ahead targets)

In [ ]:
def _tsg_to_dataset(tsg):
    tsg_len = len(tsg)
    x_shape = tsg[0][0][0].shape
    y_shape = tsg[0][1][0].shape
    xs, ys = [], []
    for seq, target in tsg:
        xs.append(seq); ys.append(target)
    xs = np.array(xs).reshape((tsg_len, *x_shape))
    ys = np.array(ys).reshape((tsg_len, *y_shape))
    return xs, ys

tsg_params = dict(length=CFG_SEQUENCE_LEN, sampling_rate=CFG_SAMPLING_RATE,
                  stride=CFG_STRIDE, batch_size=CFG_BATCH_SIZE)
train_x, train_y = _tsg_to_dataset(TimeseriesGenerator(train_n, train_n, **tsg_params))
test_x,  test_y  = _tsg_to_dataset(TimeseriesGenerator(test_n,  test_n,  **tsg_params))

print("train_x:", train_x.shape, "train_y:", train_y.shape)
print("test_x :", test_x.shape,  "test_y :", test_y.shape)
print("one window X[0] (last 3 steps):\n", train_x[0][-3:], "\n-> target y[0]:", train_y[0])

## 13. End-to-end check against `dataset.py`

The inline pipeline above should match `load_household_power()` exactly.

In [ ]:
try:
    from data.household_power.dataset import load_household_power
    (dtx, dty), (dsx, dsy) = load_household_power()
    print("load_household_power shapes:", dtx.shape, dty.shape, dsx.shape, dsy.shape)
    print("matches inline train_x:", dtx.shape == train_x.shape,
          "| matches inline test_x:", dsx.shape == test_x.shape)
except Exception as e:
    print("Import skipped (run from repo root with PYTHONPATH set):", repr(e))

## 14. Check against the GRU model

Confirm the `(N, 24, 2)` windows feed the sigmoid-output GRU and produce `(N, 2)`
one-step forecasts in `[0,1]` — the same forecaster used for network monitoring.

In [ ]:
try:
    from federation.model import build_model
    model = build_model(dataset_name="household_power", model_name="eda_check",
                        multivariate=len(FEATURES), sequence_len=CFG_SEQUENCE_LEN)
except Exception as e:
    print("Falling back to an inline GRU for the shape check:", repr(e))
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import GRU, Dense, Input
    model = Sequential([Input((CFG_SEQUENCE_LEN, len(FEATURES))),
                        GRU(64), Dense(len(FEATURES), activation="sigmoid")])
    model.compile(optimizer="adam", loss="mse")

pred = model.predict(test_x[:8], verbose=0)
print("input :", test_x[:8].shape)
print("output:", pred.shape, "| in [0,1]:", bool((pred >= 0).all() and (pred <= 1).all()))
assert pred.shape == (8, len(FEATURES)), "GRU output shape mismatch"
print("OK — household windows are model-compatible.")